# BIST 100 Model Comparison

This notebook evaluates the saved LSTM and GRU checkpoints on the untouched test period. It compares both recurrent models with persistence and a 20-day moving-average baseline using identical target dates.

The results describe historical test performance only. They are not evidence of future returns and are not financial advice.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from bist100_forecasting.data import DEFAULT_DATA_PATH, load_history
from bist100_forecasting.model_results import (
    DEFAULT_GRU_CHECKPOINT_PATH,
    DEFAULT_LSTM_CHECKPOINT_PATH,
    evaluate_saved_models,
)
from bist100_forecasting.preprocessing import DEFAULT_PROCESSED_DATA_PATH

pd.options.display.float_format = "{:,.4f}".format
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / DEFAULT_DATA_PATH
ARCHIVE_PATH = PROJECT_ROOT / DEFAULT_PROCESSED_DATA_PATH
LSTM_CHECKPOINT_PATH = PROJECT_ROOT / DEFAULT_LSTM_CHECKPOINT_PATH
GRU_CHECKPOINT_PATH = PROJECT_ROOT / DEFAULT_GRU_CHECKPOINT_PATH

## Required artifacts

Run the data preparation and both training commands before executing this notebook. Generated arrays and checkpoints remain outside version control.

In [ ]:
artifact_paths = [
    DATA_PATH,
    ARCHIVE_PATH,
    LSTM_CHECKPOINT_PATH,
    GRU_CHECKPOINT_PATH,
]
missing_paths = [path for path in artifact_paths if not path.is_file()]
if missing_paths:
    missing_names = "\n".join(f"- {path}" for path in missing_paths)
    raise FileNotFoundError(f"Missing required artifacts:\n{missing_names}")

pd.Series(
    {path.name: f"{path.stat().st_size / 1024:,.1f} KB" for path in artifact_paths},
    name="File size",
)

## Test-period evaluation

Checkpoint selection used validation loss during training. The test period is evaluated only after the best checkpoint has been restored.

In [ ]:
history = load_history(DATA_PATH)
results = evaluate_saved_models(
    history,
    archive_path=ARCHIVE_PATH,
    lstm_checkpoint_path=LSTM_CHECKPOINT_PATH,
    gru_checkpoint_path=GRU_CHECKPOINT_PATH,
)
print(f"Best method by test RMSE: {results.winner}")
results.comparison

In [ ]:
model_summary = pd.DataFrame(
    [
        {
            "Model": name,
            "Best epoch": result.best_epoch,
            "Validation loss": result.validation_loss,
            "Test MAE": result.evaluation.metrics.mae,
            "Test RMSE": result.evaluation.metrics.rmse,
            "Test MAPE (%)": result.evaluation.metrics.mape_pct,
            "Test R2": result.evaluation.metrics.r2,
        }
        for name, result in (
            ("LSTM", results.lstm),
            ("GRU", results.gru),
        )
    ]
).set_index("Model")
model_summary

## Actual and predicted closing values

The chart keeps the test observations in chronological order so prediction behavior can be inspected across the full evaluation period.

In [ ]:
predictions = pd.DataFrame(
    {
        "Actual": results.lstm.evaluation.actual,
        "LSTM": results.lstm.evaluation.predicted,
        "GRU": results.gru.evaluation.predicted,
    },
    index=results.lstm.evaluation.target_dates,
)

figure, axis = plt.subplots(figsize=(14, 6))
predictions.plot(ax=axis, linewidth=1.5)
axis.set_title("BIST 100 test-period forecasts")
axis.set_xlabel("Date")
axis.set_ylabel("Closing value")
axis.grid(alpha=0.25)
figure.tight_layout()
figure

## Recent prediction errors

Signed errors are prediction minus actual value. Positive values indicate overprediction; negative values indicate underprediction.

In [ ]:
error_frame = predictions.copy()
error_frame["LSTM error"] = error_frame["LSTM"] - error_frame["Actual"]
error_frame["GRU error"] = error_frame["GRU"] - error_frame["Actual"]
error_frame.tail(20)

## Interpretation guide

- Lower MAE, RMSE, and MAPE values indicate smaller historical prediction errors.
- Higher R² is better, but it must be considered together with error metrics and baseline results.
- A recurrent model should not be considered useful merely because it follows the overall price direction; it should outperform simple baselines on unseen dates.
- Model selection, tuning, and repeated experiments must not use the test period as training feedback.